# Proyecto final — Priorización inteligente de malla vial
Pipeline AHP + ML transversal + informe. Ver hoja de ruta y enunciado; supuestos en `SUPUESTOS.md`.

## 0. Setup
Importa rutas del proyecto y librerías.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from src.config import DATA_DIR, FEATURE_COLS, TARGET
from src.data_io import ensure_bogota_cuts, load_two_cuts
from src.features import build_training_table, score_ahp, informe_localidad
from src.train_pipeline import train_from_frames, reentrenar, load_artifact
print('ROOT', ROOT)
print('DATA_DIR', DATA_DIR)


ROOT C:\Users\DELL\OneDrive\Documentos\Proyecto Final IA\Proyecto (codigo)
DATA_DIR C:\Users\DELL\OneDrive\Documentos\Proyecto Final IA\Proyecto (codigo)\data


## 1. Datos de entrada
Asegura dos cortes IDU (descarga si faltan). El ZIP Itagüí no alcanza (un solo año / FileGDB).

In [2]:
paths = ensure_bogota_cuts(force=False)
df_a, df_b = load_two_cuts(*paths)
print('corte A', df_a.shape, paths[0].name)
print('corte B', df_b.shape, paths[1].name)
print(df_a.head(3))
print('ESTAASHTO A:')
print(df_a['ESTAASHTO'].value_counts())
print('nulos A:')
print(df_a.isna().sum())


corte A (100508, 10) bogota_estado_dic2024.csv
corte B (102238, 9) bogota_estado_jun2026.csv
   OBJECTID  PK_ID_CALZADA      CIV  LOCEST TIPOMALLAEST  INDEST  \
0         1         322232  6000769       6        Local      59   
1         2         387502  9002983       9        Local      85   
2         3         381359  9000543       9   Intermedia      82   

       ESTAASHTO  SUPERFEST  Shape__Area  Shape__Length  
0      ACEPTABLE          2   299.035555     113.867828  
1  SATISFACTORIO          2   610.692021     239.450110  
2  SATISFACTORIO          2   233.886037      85.035350  
ESTAASHTO A:
ESTAASHTO
BUENO            35199
SATISFACTORIO    25325
ACEPTABLE        11952
FALLADO          10151
POBRE             7469
MUY POBRE         5541
GRAVE             3608
SIN DATO          1263
Name: count, dtype: int64
nulos A:
OBJECTID         0
PK_ID_CALZADA    0
CIV              0
LOCEST           0
TIPOMALLAEST     0
INDEST           0
ESTAASHTO        0
SUPERFEST        0
Shape__A

## 2. EDA y calidad
Revisa INDEST inválidos, tipologías y distribución de estado.

In [3]:
print('INDEST describe A')
print(df_a['INDEST'].describe())
print('INDEST < 0:', int((df_a['INDEST'] < 0).sum()))
print(df_a['TIPOMALLAEST'].value_counts())
fig, ax = plt.subplots(figsize=(7,3))
df_a['ESTAASHTO'].value_counts().plot(kind='bar', ax=ax, color='#0E1A2B')
ax.set_title('Distribucion ESTAASHTO - corte A')
plt.tight_layout()
plt.show()


INDEST describe A


count    100508.000000
mean         66.154246
std          30.898695
min          -1.000000
25%          50.000000
50%          78.000000
75%          90.000000
max         100.000000
Name: INDEST, dtype: float64
INDEST < 0: 1263
TIPOMALLAEST
Local         60074
Intermedia    23354
Arterial      11857
Troncal        5223
Name: count, dtype: int64


C:\Users\DELL\AppData\Local\Temp\ipykernel_30724\1294915482.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Join temporal y etiqueta empeoro
Une por CIV/PK y marca deterioro AASHTO (modelo transversal, no serie de tiempo).

In [4]:
table = build_training_table(df_a, df_b)
print('filas join valido', len(table))
print('tasa empeoro', round(float(table[TARGET].mean()), 4))
print(table[TARGET].value_counts())
print(table[['segment_id','ESTAASHTO','INDEST','C_norm','J_norm','F_norm',TARGET]].head())


filas join valido 86580
tasa empeoro 0.1352
empeoro
0    74873
1    11707
Name: count, dtype: int64
  segment_id      ESTAASHTO  INDEST  C_norm  J_norm  F_norm  empeoro
0    6000769      ACEPTABLE      59    41.0    40.0    44.0        0
1    9002983  SATISFACTORIO      85    15.0    40.0    44.0        0
2    9000543  SATISFACTORIO      82    18.0    70.0    69.5        0
3    5001288          BUENO      92     8.0    40.0    44.0        0
4   11004357  SATISFACTORIO      74    26.0    40.0    44.0        0


## 4. Score AHP (reglas)
Pesos Condicion 62.5% / Jerarquia 23.9% / Flujo 13.7%.

In [5]:
table['score_ahp'] = score_ahp(table)
print(table['score_ahp'].describe())
fig, ax = plt.subplots(figsize=(6,3))
ax.hist(table['score_ahp'], bins=40, color='#FFB703', edgecolor='none')
ax.set_title('Distribucion score AHP')
plt.tight_layout()
plt.show()


count    86580.000000
mean        41.646967
std         19.074876
min         15.588000
25%         26.251500
50%         36.838000
75%         53.126500
max         99.415000
Name: score_ahp, dtype: float64


C:\Users\DELL\AppData\Local\Temp\ipykernel_30724\3176414180.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Entrenamiento ML + validacion
Pipeline sklearn, train/val/test, metricas y baseline majority. Exporta joblib.

In [6]:
result = train_from_frames(df_a, df_b)
metrics = result['metrics']
print('artifact:', result['artifact'])
print('n_rows:', result['n_rows'])
keys = ['accuracy','precision','recall','f1','roc_auc','baseline_majority_f1','positive_rate']
print({k: metrics[k] for k in keys})
print(metrics['report'])


artifact: C:\Users\DELL\OneDrive\Documentos\Proyecto Final IA\Proyecto (codigo)\models\pipeline_priorizacion.joblib
n_rows: 86580
{'accuracy': 0.9012089012089012, 'precision': 0.5846759756534192, 'recall': 0.9299544419134397, 'f1': 0.7179599912068586, 'roc_auc': 0.9686623361266784, 'baseline_majority_f1': 0.0, 'positive_rate': 0.1352159852159852}
              precision    recall  f1-score   support

           0       0.99      0.90      0.94     11231
           1       0.58      0.93      0.72      1756

    accuracy                           0.90     12987
   macro avg       0.79      0.91      0.83     12987
weighted avg       0.93      0.90      0.91     12987



## 6. Score final e informe
Combina reglas (70%) + P_empeora (30%).

In [7]:
art = load_artifact()
scored = pd.read_csv(DATA_DIR / 'scored_segments.csv')
print(scored[['segment_id','P_empeora','score_ahp','score_final']].head())
print(informe_localidad(scored, localidad=None, top_n=8))


   segment_id  P_empeora  score_ahp  score_final
0     6000769   0.193318    41.2130    34.648644
1     9002983   0.020249    24.9630    18.081581
2     9000543   0.027640    37.5015    27.080262
3     5001288   0.002417    20.5880    14.484106
4    11004357   0.512354    31.8380    37.657232


Informe de priorización — top 8 segmentos
Score medio: 80.1 | P(empeora) media: 0.458
Criterio: 70% reglas AHP (condición/jerarquía/flujo) + 30% ML transversal.
Segmentos prioritarios:
- ID 16004658: score=81.5, estado=FALLADO, P_empeora=0.397, C=100, J=100, F=95
- ID 8010885: score=80.5, estado=GRAVE, P_empeora=0.537, C=88, J=100, F=95
- ID 6000690: score=80.3, estado=GRAVE, P_empeora=0.532, C=88, J=100, F=95
- ID 7005343: score=80.2, estado=GRAVE, P_empeora=0.515, C=89, J=100, F=95
- ID 7007590: score=80.1, estado=GRAVE, P_empeora=0.525, C=88, J=100, F=95
- ID 7008253: score=79.5, estado=FALLADO, P_empeora=0.329, C=100, J=100, F=95
- ID 16004878: score=79.4, estado=FALLADO, P_empeora=0.327, C=100, J=100, F=95
- ID 5006901: score=79.4, estado=GRAVE, P_empeora=0.501, C=88, J=100, F=95
Limitaciones: Bitcarrier/Waze/SPT-CPT pueden estar ausentes; DTM descartado por precisión vertical insuficiente.


## 7. reentrenar(ruta_datos)
API para otras ciudades (ejemplo con carpeta data/).

In [8]:
info = reentrenar(DATA_DIR)
print('reentrenar OK', info['n_rows'], 'F1', info['metrics']['f1'])


reentrenar OK 86580 F1 0.7179599912068586


## 8. Conclusiones tecnicas
- Datos suficientes via IDU abierto; Itagui pendiente de CSV canonico.
- ML transversal viable; DTM fuera; SPT/CPT sin fuente publica.
- App: streamlit run app.py